# exp10 — Attention Pattern Distance at Acrostic Positions

**Mentor hypothesis (June 15 meeting):**
> *Attention patterns → функция потерь (KL divergence, cosine similarity, MSE) vs обычный паттерн*

**Design:**

For each sentence-start token (acrostic position), we extract the attention weight distribution over all preceding tokens — what the model attends TO at the moment it writes the constrained letter.

Two metrics:

1. **Entropy** of attention at sentence-start positions: stego vs open (cross-condition, no alignment needed)
2. **Within-sequence KL divergence**: KL(attention at sentence-start ∥ mean attention at all other positions) — how different are acrostic positions from the "usual" pattern *within the same sequence*

Both metrics are computed per layer and per head (32×32 grid).

**Prediction (if stego forces extra focused attention on instruction):**  
- Lower entropy at sentence-starts in stego vs open
- Higher within-sequence KL in stego vs open

**Null hypothesis:** no difference — same structural property as any sentence boundary.

**Data:** `results/exp01/valid_pairs.json`, fidelity=1.0 subset (~159 pairs), Llama-3.1-8B-Instruct.

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))

    if not os.path.exists('stego_CoT'):
        os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
    os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate scipy')

MODEL_ID   = 'meta-llama/Llama-3.1-8B-Instruct'
INPUT_DIR  = 'results/exp01'
OUTPUT_DIR = 'results/exp10'
N_MAX      = 80   # same as exp04-exp08 for comparability

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('cwd:', os.getcwd())

# Runtime check — A100 (40GB) required. T4/V100 (16GB) will OOM.
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  ({mem:.0f} GB)')
    if mem < 30:
        print('WARNING: <30 GB VRAM. Switch to A100: Runtime → Change runtime type → A100')
        print('Llama-3.1-8B + output_attentions=True needs ~20 GB peak.')
else:
    print('WARNING: no GPU detected')

In [ ]:
import json
import gc
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import entropy as scipy_entropy
from transformers import AutoModelForCausalLM, AutoTokenizer

with open(f'{INPUT_DIR}/valid_pairs.json') as f:
    all_pairs = json.load(f)

pairs = [p for p in all_pairs if p['fidelity'] == 1.0]
pairs = pairs[:N_MAX]
print(f'Total pairs: {len(all_pairs)},  fidelity=1.0: {len([p for p in all_pairs if p["fidelity"]==1.0])}')
print(f'Using: {len(pairs)} pairs')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',  # required: Flash Attention / SDPA break output_attentions=True
)
model.eval()

N_LAYERS = model.config.num_hidden_layers
N_HEADS  = model.config.num_attention_heads
DEVICE   = next(model.parameters()).device

print(f'Layers: {N_LAYERS},  heads: {N_HEADS}')
print(f'attn_implementation: eager (required for output_attentions=True)')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def find_sentence_starts(token_ids, plen):
    """Return absolute token positions of sentence-start tokens in CoT.
    Reused verbatim from exp08 for consistency."""
    cot_ids   = list(token_ids[plen:])
    full_text = tokenizer.decode(cot_ids, skip_special_tokens=True)

    char_starts = []
    i = 0
    while i < len(full_text) and full_text[i] == '\n':
        i += 1
    if i < len(full_text):
        char_starts.append(i)
    while i < len(full_text):
        if full_text[i] == '\n':
            j = i + 1
            while j < len(full_text) and full_text[j] == '\n':
                j += 1
            if j - i >= 2 and j < len(full_text):
                char_starts.append(j)
            i = max(i + 1, j)
        else:
            i += 1

    positions, found, prev_len = [], 0, 0
    for tok_idx in range(len(cot_ids)):
        curr_len = len(tokenizer.decode(cot_ids[:tok_idx + 1], skip_special_tokens=True))
        while found < len(char_starts) and prev_len <= char_starts[found] < curr_len:
            positions.append(plen + tok_idx)
            found += 1
        prev_len = curr_len
        if found >= len(char_starts):
            break
    return positions


@torch.no_grad()
def get_attention_two_sets(token_ids, sent_positions, other_positions):
    """ONE forward pass. Returns attention rows for two position sets.

    sent_positions:  list of absolute positions (sentence-starts)
    other_positions: list of absolute positions (other CoT tokens)

    Returns:
        sent_attn:  {layer_idx -> list of np.array [n_heads, p+1]}
        other_attn: {layer_idx -> list of np.array [n_heads, p+1]}
    """
    all_pos = sorted(set(sent_positions) | set(other_positions))
    sent_set  = set(sent_positions)
    other_set = set(other_positions)

    ids_t = torch.tensor([token_ids], dtype=torch.long).to(DEVICE)
    out   = model(ids_t, output_attentions=True, use_cache=False)

    sent_attn  = {l: [] for l in range(N_LAYERS)}
    other_attn = {l: [] for l in range(N_LAYERS)}

    for layer_idx, layer_attn in enumerate(out.attentions):
        # layer_attn: [1, n_heads, seq_len, seq_len]
        seq_len = layer_attn.shape[2]
        for p in all_pos:
            if p >= seq_len:
                continue
            row = layer_attn[0, :, p, :p + 1].cpu().float().numpy()  # [n_heads, p+1]
            if p in sent_set:
                sent_attn[layer_idx].append(row)
            if p in other_set:
                other_attn[layer_idx].append(row)

    del out, ids_t
    torch.cuda.empty_cache()
    gc.collect()
    return sent_attn, other_attn


def attention_entropy(attn_row):
    """Shannon entropy of attention distribution. attn_row: [n_heads, L]"""
    entropies = []
    for h in range(attn_row.shape[0]):
        p = attn_row[h].astype(float) + 1e-12
        p /= p.sum()
        entropies.append(scipy_entropy(p, base=2))
    return np.array(entropies)  # [n_heads]


def within_kl(sent_rows, other_rows):
    """KL(sent_start || mean_other) per head, averaged over sentence-starts.

    Uses min available length across all rows for fair cross-condition comparison.
    """
    if not sent_rows or not other_rows:
        return np.zeros(N_HEADS)

    min_len = min(
        min(r.shape[1] for r in sent_rows),
        min(r.shape[1] for r in other_rows),
    )
    if min_len < 2:
        return np.zeros(N_HEADS)

    other_stack = np.stack([r[:, :min_len] for r in other_rows], axis=0)
    usual = other_stack.mean(axis=0)  # [n_heads, min_len]

    kl_vals = np.zeros(N_HEADS)
    for row in sent_rows:
        for h in range(N_HEADS):
            p = row[h, :min_len].astype(float) + 1e-12
            q = usual[h].astype(float) + 1e-12
            p /= p.sum()
            q /= q.sum()
            kl_vals[h] += float(scipy_entropy(p, q))

    return kl_vals / len(sent_rows)

In [ ]:
# Run this cell ONCE to clear stale checkpoint before re-running after bug fix.
# Comment out after first run.
import pathlib
ckpt = pathlib.Path(f'{OUTPUT_DIR}/exp10_raw.json')
if ckpt.exists():
    ckpt.unlink()
    print('Cleared stale checkpoint — ready for clean run.')
else:
    print('No checkpoint to clear.')

In [ ]:
CKPT = f'{OUTPUT_DIR}/exp10_raw.json'

if os.path.exists(CKPT):
    with open(CKPT) as f:
        saved = json.load(f)
    results = saved['results']
    skipped = saved['skipped']
    print(f'Resumed from checkpoint: {len(results)} pairs done, {skipped} skipped')
else:
    results, skipped = [], 0

done_idx = {r['pair_idx'] for r in results}

for i, pair in enumerate(pairs):
    if i in done_idx:
        continue

    try:
        stego_ids = pair['stego_ids']
        open_ids  = pair['open_ids']
        splen     = pair['stego_plen']
        oplen     = pair['open_plen']

        # sentence-start positions (limit to payload length)
        s_starts = find_sentence_starts(stego_ids, splen)[:len(pair['payload'])]
        o_starts = find_sentence_starts(open_ids,  oplen)[:len(pair['payload'])]

        if not s_starts or not o_starts:
            skipped += 1
            continue

        # sample up to 20 other (non-sentence-start) CoT positions
        def sample_other(ids, plen, starts, k=20):
            all_pos = list(range(plen, len(ids)))
            other   = sorted(set(all_pos) - set(starts))
            step    = max(1, len(other) // k)
            return other[::step][:k]

        s_other = sample_other(stego_ids, splen, s_starts)
        o_other = sample_other(open_ids,  oplen, o_starts)

        # 2 forward passes (stego + open), each extracting sent AND other positions
        s_sent_attn, s_other_attn = get_attention_two_sets(stego_ids, s_starts, s_other)
        o_sent_attn, o_other_attn = get_attention_two_sets(open_ids,  o_starts, o_other)

        # compute metrics per layer
        pair_rec = {
            'pair_idx':    i,
            'n_s_starts':  len(s_starts),
            'n_o_starts':  len(o_starts),
        }

        for layer in range(N_LAYERS):
            s_ent = (np.mean([attention_entropy(r) for r in s_sent_attn[layer]], axis=0)
                     if s_sent_attn[layer] else np.zeros(N_HEADS))
            o_ent = (np.mean([attention_entropy(r) for r in o_sent_attn[layer]], axis=0)
                     if o_sent_attn[layer] else np.zeros(N_HEADS))
            s_kl  = within_kl(s_sent_attn[layer], s_other_attn[layer])
            o_kl  = within_kl(o_sent_attn[layer], o_other_attn[layer])

            pair_rec[f'L{layer}'] = {
                's_ent': s_ent.tolist(),
                'o_ent': o_ent.tolist(),
                's_kl':  s_kl.tolist(),
                'o_kl':  o_kl.tolist(),
            }

        results.append(pair_rec)

        if (i + 1) % 5 == 0 or i == len(pairs) - 1:
            with open(CKPT, 'w') as f:
                json.dump({'results': results, 'skipped': skipped}, f)
            print(f'[{i+1}/{len(pairs)}] processed={len(results)}, skipped={skipped}')

    except Exception as e:
        print(f'  pair {i} error: {e}')
        skipped += 1

with open(CKPT, 'w') as f:
    json.dump({'results': results, 'skipped': skipped}, f)
print(f'\nDone: {len(results)} pairs, {skipped} skipped')

In [ ]:
# Reload if coming from fresh session
if 'results' not in locals() or not results:
    with open(CKPT) as f:
        d = json.load(f)
    results, skipped = d['results'], d['skipped']
    print(f'Loaded {len(results)} pairs')

n = len(results)
print(f'Analysing {n} pairs, {skipped} skipped')

# Aggregate: mean per layer per head across all pairs
s_ent_mean = np.zeros((N_LAYERS, N_HEADS))
o_ent_mean = np.zeros((N_LAYERS, N_HEADS))
s_kl_mean  = np.zeros((N_LAYERS, N_HEADS))
o_kl_mean  = np.zeros((N_LAYERS, N_HEADS))

for rec in results:
    for layer in range(N_LAYERS):
        key = f'L{layer}'
        if key not in rec:
            continue
        s_ent_mean[layer] += np.array(rec[key]['s_ent'])
        o_ent_mean[layer] += np.array(rec[key]['o_ent'])
        s_kl_mean[layer]  += np.array(rec[key]['s_kl'])
        o_kl_mean[layer]  += np.array(rec[key]['o_kl'])

s_ent_mean /= n
o_ent_mean /= n
s_kl_mean  /= n
o_kl_mean  /= n

# Difference maps
ent_diff = s_ent_mean - o_ent_mean  # negative = stego more focused
kl_diff  = s_kl_mean  - o_kl_mean   # positive = stego deviates more from usual

# Per-layer summary (mean over heads)
print('\n=== Entropy (stego − open) per layer, mean over heads ===')
print('Negative = stego has more focused (lower entropy) attention at acrostic positions')
for layer in range(N_LAYERS):
    d_val = ent_diff[layer].mean()
    bar = '█' * int(abs(d_val) * 20) if abs(d_val) > 0 else ''
    sign = '-' if d_val < 0 else '+'
    print(f'  L{layer:02d}: {sign}{abs(d_val):.4f}  {bar}')

print('\n=== Within-sequence KL (stego − open) per layer, mean over heads ===')
print('Positive = stego sentence-starts deviate more from "usual" attention pattern')
for layer in range(N_LAYERS):
    d_val = kl_diff[layer].mean()
    bar = '█' * int(abs(d_val) * 100) if abs(d_val) > 0 else ''
    sign = '+' if d_val > 0 else '-'
    print(f'  L{layer:02d}: {sign}{abs(d_val):.4f}  {bar}')

# Top heads by KL difference
flat = kl_diff.flatten()
top5 = np.argsort(flat)[-5:][::-1]
print('\n=== Top 5 (layer, head) by KL difference ===')
for idx in top5:
    l, h = divmod(idx, N_HEADS)
    print(f'  L{l:02d} H{h:02d}: KL_diff={kl_diff[l,h]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── Entropy heatmaps ──────────────────────────────────────────────────────────
for ax, (data, title) in zip(axes[0], [
    (s_ent_mean, 'Entropy at acrostic positions — STEGO'),
    (o_ent_mean, 'Entropy at acrostic positions — OPEN'),
]):
    im = ax.imshow(data, aspect='auto', cmap='viridis')
    ax.set_xlabel('Head')
    ax.set_ylabel('Layer')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Entropy (bits)')

# ── Difference heatmaps ────────────────────────────────────────────────────────
vmax_ent = np.abs(ent_diff).max()
im3 = axes[1, 0].imshow(ent_diff, aspect='auto', cmap='RdBu_r',
                         vmin=-vmax_ent, vmax=vmax_ent)
axes[1, 0].set_xlabel('Head')
axes[1, 0].set_ylabel('Layer')
axes[1, 0].set_title('Entropy difference (stego − open)\nNegative = stego more focused')
plt.colorbar(im3, ax=axes[1, 0], label='Δ Entropy (bits)')

vmax_kl = np.abs(kl_diff).max()
im4 = axes[1, 1].imshow(kl_diff, aspect='auto', cmap='RdBu_r',
                          vmin=-vmax_kl, vmax=vmax_kl)
axes[1, 1].set_xlabel('Head')
axes[1, 1].set_ylabel('Layer')
axes[1, 1].set_title('Within-seq KL difference (stego − open)\nPositive = stego deviates more from usual')
plt.colorbar(im4, ax=axes[1, 1], label='Δ KL (nats)')

plt.suptitle(
    f'Attention pattern metrics at sentence-start (acrostic) positions\n'
    f'n={n} pairs (fidelity=1.0), Llama-3.1-8B-Instruct',
    fontsize=13
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp10_attention_heatmaps.png', dpi=150)
plt.show()

# ── Per-layer bar chart (mean over heads) ─────────────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5))

layers = np.arange(N_LAYERS)
axes2[0].bar(layers, ent_diff.mean(axis=1), color=[
    'steelblue' if v >= 0 else 'crimson' for v in ent_diff.mean(axis=1)
])
axes2[0].axhline(0, color='black', linewidth=1)
axes2[0].set_xlabel('Layer')
axes2[0].set_ylabel('Entropy difference (stego − open)')
axes2[0].set_title('Attention entropy: stego − open at acrostic positions\n(< 0 = stego more focused)')

axes2[1].bar(layers, kl_diff.mean(axis=1), color=[
    'crimson' if v >= 0 else 'steelblue' for v in kl_diff.mean(axis=1)
])
axes2[1].axhline(0, color='black', linewidth=1)
axes2[1].set_xlabel('Layer')
axes2[1].set_ylabel('KL difference (stego − open)')
axes2[1].set_title('Within-seq KL: stego − open at acrostic positions\n(> 0 = stego deviates more from usual)')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp10_per_layer_bars.png', dpi=150)
plt.show()
print('Plots saved.')

In [ ]:
# Statistical test: is the overall difference significant?
# Per-pair mean entropy and KL across all layers and heads

s_ent_per_pair = []
o_ent_per_pair = []
s_kl_per_pair  = []
o_kl_per_pair  = []

for rec in results:
    s_ent_vals, o_ent_vals, s_kl_vals, o_kl_vals = [], [], [], []
    for layer in range(N_LAYERS):
        key = f'L{layer}'
        if key in rec:
            s_ent_vals.extend(rec[key]['s_ent'])
            o_ent_vals.extend(rec[key]['o_ent'])
            s_kl_vals.extend(rec[key]['s_kl'])
            o_kl_vals.extend(rec[key]['o_kl'])
    s_ent_per_pair.append(np.mean(s_ent_vals))
    o_ent_per_pair.append(np.mean(o_ent_vals))
    s_kl_per_pair.append(np.mean(s_kl_vals))
    o_kl_per_pair.append(np.mean(o_kl_vals))

s_ent_arr = np.array(s_ent_per_pair)
o_ent_arr = np.array(o_ent_per_pair)
s_kl_arr  = np.array(s_kl_per_pair)
o_kl_arr  = np.array(o_kl_per_pair)

t_ent, p_ent = stats.ttest_rel(s_ent_arr, o_ent_arr)
t_kl,  p_kl  = stats.ttest_rel(s_kl_arr,  o_kl_arr)

print('=== Paired t-test: stego vs open, mean over all layers and heads ===')
print(f'Entropy:   stego={s_ent_arr.mean():.4f}  open={o_ent_arr.mean():.4f}  '
      f'diff={s_ent_arr.mean()-o_ent_arr.mean():.4f}  t={t_ent:.3f}  p={p_ent:.4f}')
print(f'Within KL: stego={s_kl_arr.mean():.4f}  open={o_kl_arr.mean():.4f}  '
      f'diff={s_kl_arr.mean()-o_kl_arr.mean():.4f}  t={t_kl:.3f}  p={p_kl:.4f}')
print()
print('Interpretation:')
if p_ent < 0.05 and s_ent_arr.mean() < o_ent_arr.mean():
    print('  Entropy: stego attention is MORE FOCUSED at acrostic positions (p<0.05)')
elif p_ent < 0.05:
    print('  Entropy: significant difference, but direction unexpected')
else:
    print('  Entropy: no significant difference (p>=0.05)')

if p_kl < 0.05 and s_kl_arr.mean() > o_kl_arr.mean():
    print('  KL: stego sentence-starts DEVIATE MORE from usual attention (p<0.05)')
elif p_kl < 0.05:
    print('  KL: significant difference, but direction unexpected')
else:
    print('  KL: no significant difference (p>=0.05)')

In [ ]:
summary = {
    'model':   MODEL_ID,
    'n_pairs': n,
    'skipped': skipped,
    'n_layers': N_LAYERS,
    'n_heads':  N_HEADS,
    'entropy': {
        'stego_mean': round(float(s_ent_arr.mean()), 6),
        'open_mean':  round(float(o_ent_arr.mean()), 6),
        'diff':       round(float(s_ent_arr.mean() - o_ent_arr.mean()), 6),
        'ttest': {'t': round(float(t_ent), 3), 'p': round(float(p_ent), 6)},
    },
    'within_kl': {
        'stego_mean': round(float(s_kl_arr.mean()), 6),
        'open_mean':  round(float(o_kl_arr.mean()), 6),
        'diff':       round(float(s_kl_arr.mean() - o_kl_arr.mean()), 6),
        'ttest': {'t': round(float(t_kl), 3), 'p': round(float(p_kl), 6)},
    },
    'top5_kl_diff_heads': [
        {'layer': int(l), 'head': int(h), 'kl_diff': round(float(kl_diff[l, h]), 6)}
        for idx in np.argsort(kl_diff.flatten())[-5:][::-1]
        for l, h in [divmod(int(idx), N_HEADS)]
    ],
}

import json
out_path = f'{OUTPUT_DIR}/exp10_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)
print(json.dumps(summary, indent=2))

# ── Google Drive backup ───────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    import shutil, pathlib
    drive.mount('/content/drive')
    dst = pathlib.Path('/content/drive/MyDrive/stego_cot_results/exp10')
    dst.mkdir(parents=True, exist_ok=True)
    for fp in pathlib.Path(OUTPUT_DIR).iterdir():
        shutil.copy(fp, dst / fp.name)
        print(f'  -> Drive: {fp.name}')